<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part A: Foundations and Data Exploration</h2>
<h2>Notebook A01: Loading and Manipulating the Data</h2>
</div>

This notebook covers the basics you need before moving on to visualisation and more advanced analysis: what a time series looks like in pandas, how to load one, and the handful of operations that come up in almost every project.

---

**Contents**

1. [Imports](#1.-Imports)
2. [Loading the Data](#2.-Loading-the-Data)
3. [Wide, Long, and Compact Formats](#3.-Wide,-Long,-and-Compact-Formats)
4. [Resampling](#4.-Resampling)
5. [Indexing and Slicing](#5.-Indexing-and-Slicing)
6. [Rolling Windows](#6.-Rolling-Windows)

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>1. Imports</h3>
</div>

We only need `pandas` for this notebook. `nb_config` gives us the dataset paths so we never need to hard-code them.

> **Datetime handling.** This notebook focuses on time series operations, not on parsing dates from raw strings. If you need a refresher on that, the resources below are a good starting point.
>
> | Source | Reference | Notes |
> | ------ | --------- | ----- |
> | [GitHub](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks_v1) | Python Data Science Handbook (Jake VanderPlas) | Notebook 3.11 focuses specifically on time series in pandas. |
> | [GitHub](https://github.com/PacktPublishing/Modern-Time-Series-Forecasting-with-Python-2E/tree/main/notebooks/Chapter02) | Modern Time Series Forecasting with Python, 2nd ed. (Manu Joseph) | Chapter 2 walks through the basics on a real dataset. |
> | [Kaggle](https://www.kaggle.com/code/parulpandey/getting-started-with-time-series-using-pandas) | Getting Started with Time Series Using Pandas (Parul Pandey) | A practical notebook covering the most common datetime operations. |
> | [Medium](https://medium.com/@noorfatimaafzalbutt/working-with-dates-and-times-in-pandas-a-comprehensive-guide-fda47929ace4) | Working with Dates and Times in Pandas (Noor Fatima) | A code-heavy guide to pandas datetime features. |

In [1]:
import pandas as pd

import nb_config

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>2. Loading the Data</h3>
</div>

We use the CDC regional air temperature dataset throughout this notebook. It contains monthly mean air temperatures for 17 German regions from 1881 to the present, sourced from the [Deutscher Wetterdienst](https://www.dwd.de). Each column is a region; each row is one month.

> Each notebook uses one or two datasets as worked examples. You are free to follow along with those or swap in any dataset you prefer.

If you have not yet downloaded and prepared the dataset, run [F01b - Preparing the CDC dataset](../notebooks/F01b_Preparing_CDC_dataset.ipynb) first.

In [2]:
df = pd.read_parquet(nb_config.CDC_TEMP_PATH)
df.head()

,Brandenburg/Berlin,Brandenburg,Baden-Wuerttemberg,Bayern,Hessen,Mecklenburg-Vorpommern,Niedersachsen,Niedersachsen/Hamburg/Bremen,Nordrhein-Westfalen,Rheinland-Pfalz,Schleswig-Holstein,Saarland,Sachsen,Sachsen-Anhalt,Thueringen/Sachsen-Anhalt,Thueringen,Deutschland
date,,,,,,,,,,,,,,,,,
1881-01-01,-5.54,-5.56,-4.89,-6.51,-5.68,-5.07,-4.55,-4.55,-4.21,-4.49,-4.06,-4.15,-6.22,-5.89,-6.28,-6.76,-5.36
1881-02-01,-1.00,-1.01,1.33,-0.73,0.75,-2.01,-0.05,-0.07,1.64,1.45,-1.81,2.25,-0.92,-0.42,-0.34,-0.25,-0.10
1881-03-01,1.71,1.70,4.42,2.50,3.43,0.40,2.20,2.19,3.81,4.49,0.54,5.37,1.79,2.20,2.18,2.14,2.61
1881-04-01,5.56,5.55,6.63,5.01,5.88,4.63,5.64,5.64,6.44,6.65,5.02,7.08,4.41,5.51,5.04,4.43,5.55
1881-05-01,12.85,12.83,11.44,10.99,12.15,11.58,12.03,12.03,12.55,12.25,11.16,12.38,11.49,12.57,12.05,11.40,11.81


#### The DatetimeIndex

Notice that the index is a `DatetimeIndex`. This is the standard way to work with time series in pandas. It is what enables time-aware operations like resampling, slicing by date string, and rolling windows. If you load a dataset and the index is still a plain integer or a string, the first thing to do is convert it:

```python
df.index = pd.to_datetime(df.index)
```

Or, if the date is in a column rather than the index:

```python
df = df.set_index(pd.to_datetime(df["date_column"]))
```

Our CDC dataset already has a proper `DatetimeIndex` after the preparation step, so we can get straight to work.

In [3]:
# Shape, dtypes, and missing value count at a glance
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1736 entries, 1881-01-01 to 2025-08-01
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Brandenburg/Berlin            1736 non-null   float64
 1   Brandenburg                   1736 non-null   float64
 2   Baden-Wuerttemberg            1736 non-null   float64
 3   Bayern                        1736 non-null   float64
 4   Hessen                        1736 non-null   float64
 5   Mecklenburg-Vorpommern        1736 non-null   float64
 6   Niedersachsen                 1736 non-null   float64
 7   Niedersachsen/Hamburg/Bremen  1736 non-null   float64
 8   Nordrhein-Westfalen           1736 non-null   float64
 9   Rheinland-Pfalz               1736 non-null   float64
 10  Schleswig-Holstein            1736 non-null   float64
 11  Saarland                      1736 non-null   float64
 12  Sachsen                       1736 non-null 

In [4]:
# Summary statistics for all regions
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Brandenburg/Berlin,1736.0,8.887967,7.020166,-11.50,2.8000,8.665,15.3100,23.22
Brandenburg,1736.0,8.875916,7.018599,-11.52,2.7875,8.650,15.3000,23.19
Baden-Wuerttemberg,1736.0,8.307817,6.636643,-10.19,2.6075,8.135,14.4075,21.78
Bayern,1736.0,7.680351,6.932772,-11.28,1.6375,7.705,14.1300,21.23
Hessen,1736.0,8.420714,6.505639,-9.58,2.8500,8.250,14.4000,21.82
Mecklenburg-Vorpommern,1736.0,8.379862,6.622568,-11.14,2.7200,8.000,14.5225,21.93
Niedersachsen,1736.0,8.819798,6.220485,-9.09,3.5675,8.545,14.5125,22.07
Niedersachsen/Hamburg/Bremen,1736.0,8.823249,6.220985,-9.08,3.5700,8.550,14.5200,22.08
Nordrhein-Westfalen,1736.0,9.099770,6.066321,-8.08,3.9750,8.830,14.6250,22.28
Rheinland-Pfalz,1736.0,8.766705,6.392545,-9.12,3.2375,8.540,14.6575,22.59


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>3. Wide, Long, and Compact Formats</h3>
</div>

When a dataset contains several time series, there are a few common ways to organise it. You will run into all three in practice, so it is worth knowing how to convert between them.

| Format | Structure | Typical use |
|--------|-----------|-------------|
| **Wide** | One row per timestamp, one column per series | Most pandas operations, plotting |
| **Long** | One row per (timestamp, series) pair | Seaborn, some ML libraries, databases |
| **Compact** | One row per series, with the full value array stored as a single cell | Some specialised forecasting libraries |

Our CDC dataset is in wide format. We will use it to show all the conversions.

#### Wide to long

In [5]:
df_long = pd.melt(
    df,
    ignore_index=False,       # keep the DatetimeIndex
    value_vars=df.columns,
    var_name="region",
    value_name="temperature",
)
df_long.index.name = "date"
df_long = df_long.sort_index()
df_long.head(10)

,region,temperature
date,,
1881-01-01,Brandenburg/Berlin,-5.54
1881-01-01,Hessen,-5.68
1881-01-01,Baden-Wuerttemberg,-4.89
1881-01-01,Thueringen,-6.76
1881-01-01,Sachsen-Anhalt,-5.89
1881-01-01,Schleswig-Holstein,-4.06
1881-01-01,Niedersachsen/Hamburg/Bremen,-4.55
1881-01-01,Niedersachsen,-4.55
1881-01-01,Saarland,-4.15


#### Long to wide

`pivot_table` reverses the melt. Each unique value in the `region` column becomes a column again.

In [6]:
df_wide_from_long = df_long.pivot_table(
    index=df_long.index,
    columns="region",
    values="temperature",
)
df_wide_from_long.columns.name = None   # drop the 'region' label from the column axis
df_wide_from_long.head()

,Baden-Wuerttemberg,Bayern,Brandenburg,Brandenburg/Berlin,Deutschland,Hessen,Mecklenburg-Vorpommern,Niedersachsen,Niedersachsen/Hamburg/Bremen,Nordrhein-Westfalen,Rheinland-Pfalz,Saarland,Sachsen,Sachsen-Anhalt,Schleswig-Holstein,Thueringen,Thueringen/Sachsen-Anhalt
date,,,,,,,,,,,,,,,,,
1881-01-01,-4.89,-6.51,-5.56,-5.54,-5.36,-5.68,-5.07,-4.55,-4.55,-4.21,-4.49,-4.15,-6.22,-5.89,-4.06,-6.76,-6.28
1881-02-01,1.33,-0.73,-1.01,-1.00,-0.10,0.75,-2.01,-0.05,-0.07,1.64,1.45,2.25,-0.92,-0.42,-1.81,-0.25,-0.34
1881-03-01,4.42,2.50,1.70,1.71,2.61,3.43,0.40,2.20,2.19,3.81,4.49,5.37,1.79,2.20,0.54,2.14,2.18
1881-04-01,6.63,5.01,5.55,5.56,5.55,5.88,4.63,5.64,5.64,6.44,6.65,7.08,4.41,5.51,5.02,4.43,5.04
1881-05-01,11.44,10.99,12.83,12.85,11.81,12.15,11.58,12.03,12.03,12.55,12.25,12.38,11.49,12.57,11.16,11.40,12.05


#### Compact format

Some forecasting libraries (e.g. GluonTS, certain Darts loaders) expect a compact layout: one row per series, where each row stores the metadata (start timestamp, frequency, number of observations) alongside the actual values as a single array. This is less common for data exploration but useful to recognise.

```bash
             Start       Frequency  n_Elements  Values
Deutschland  1881-01-01  1MS        1736        [-5.36, -2.73, ...]
Bayern       1881-01-01  1MS        1736        [-6.51, -3.62, ...]
...
```

The conversion from wide to compact is straightforward but we will not need it in this course until Part D. We show it here for completeness.

In [7]:
df_compact = pd.DataFrame(
    {
        "Start": df.apply(lambda col: col.first_valid_index()),
        "Frequency": "1MS",
        "n_Elements": df.shape[0],
        "Values": [df[col].values for col in df.columns],
    },
    index=df.columns,
)
df_compact

,Start,Frequency,n_Elements,Values
Brandenburg/Berlin,1881-01-01,1MS,1736,"[-5.54, -1.0, 1.71, 5.56, 12.85, 15.73, 19.19,..."
Brandenburg,1881-01-01,1MS,1736,"[-5.56, -1.01, 1.7, 5.55, 12.83, 15.71, 19.16,..."
Baden-Wuerttemberg,1881-01-01,1MS,1736,"[-4.89, 1.33, 4.42, 6.63, 11.44, 15.3, 19.08, ..."
Bayern,1881-01-01,1MS,1736,"[-6.51, -0.73, 2.5, 5.01, 10.99, 14.65, 18.4, ..."
Hessen,1881-01-01,1MS,1736,"[-5.68, 0.75, 3.43, 5.88, 12.15, 15.3, 18.77, ..."
Mecklenburg-Vorpommern,1881-01-01,1MS,1736,"[-5.07, -2.01, 0.4, 4.63, 11.58, 15.08, 17.92,..."
Niedersachsen,1881-01-01,1MS,1736,"[-4.55, -0.05, 2.2, 5.64, 12.03, 14.9, 18.4, 1..."
Niedersachsen/Hamburg/Bremen,1881-01-01,1MS,1736,"[-4.55, -0.07, 2.19, 5.64, 12.03, 14.9, 18.4, ..."
Nordrhein-Westfalen,1881-01-01,1MS,1736,"[-4.21, 1.64, 3.81, 6.44, 12.55, 15.37, 19.04,..."
Rheinland-Pfalz,1881-01-01,1MS,1736,"[-4.49, 1.45, 4.49, 6.65, 12.25, 15.56, 19.01,..."


And back to wide:

In [8]:
series_dict = {
    series_name: pd.Series(
        data=row["Values"],
        index=pd.date_range(start=row["Start"], periods=row["n_Elements"], freq=row["Frequency"]),
    )
    for series_name, row in df_compact.iterrows()
}

df_wide_from_compact = pd.DataFrame(series_dict)
df_wide_from_compact.index.name = "date"
df_wide_from_compact.head()

,Brandenburg/Berlin,Brandenburg,Baden-Wuerttemberg,Bayern,Hessen,Mecklenburg-Vorpommern,Niedersachsen,Niedersachsen/Hamburg/Bremen,Nordrhein-Westfalen,Rheinland-Pfalz,Schleswig-Holstein,Saarland,Sachsen,Sachsen-Anhalt,Thueringen/Sachsen-Anhalt,Thueringen,Deutschland
date,,,,,,,,,,,,,,,,,
1881-01-01,-5.54,-5.56,-4.89,-6.51,-5.68,-5.07,-4.55,-4.55,-4.21,-4.49,-4.06,-4.15,-6.22,-5.89,-6.28,-6.76,-5.36
1881-02-01,-1.00,-1.01,1.33,-0.73,0.75,-2.01,-0.05,-0.07,1.64,1.45,-1.81,2.25,-0.92,-0.42,-0.34,-0.25,-0.10
1881-03-01,1.71,1.70,4.42,2.50,3.43,0.40,2.20,2.19,3.81,4.49,0.54,5.37,1.79,2.20,2.18,2.14,2.61
1881-04-01,5.56,5.55,6.63,5.01,5.88,4.63,5.64,5.64,6.44,6.65,5.02,7.08,4.41,5.51,5.04,4.43,5.55
1881-05-01,12.85,12.83,11.44,10.99,12.15,11.58,12.03,12.03,12.55,12.25,11.16,12.38,11.49,12.57,12.05,11.40,11.81


**Exercise.** Convert `df_long` directly to compact format without going through wide first. Each group in `df_long` corresponds to one series.

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>4. Resampling</h3>
</div>

Resampling changes the frequency of a time series.

**Downsampling** aggregates multiple observations into a single one, for example, going from monthly to yearly by taking the mean. You can use any aggregation function: `mean`, `sum`, `min`, `max`, `median`.

**Upsampling** goes the other way, introducing new time points between existing ones. The new points start out as `NaN` and need to be filled in. We cover the filling strategies in the missing data notebook.

The `resample` method takes a frequency alias string. Some common ones:

| Alias | Frequency |
|-------|-----------|
| `"h"` | Hourly |
| `"D"` | Daily |
| `"W"` | Weekly |
| `"MS"` | Month start |
| `"QS"` | Quarter start |
| `"YS"` | Year start |

In [9]:
# Downsample to quarterly frequency — mean temperature per quarter
quarter_df = df.resample("QS").mean()
print(f"Monthly shape:   {df.shape}")
print(f"Quarterly shape: {quarter_df.shape}")
quarter_df.head()

Monthly shape:   (1736, 17)
Quarterly shape: (579, 17)


,Brandenburg/Berlin,Brandenburg,Baden-Wuerttemberg,Bayern,Hessen,Mecklenburg-Vorpommern,Niedersachsen,Niedersachsen/Hamburg/Bremen,Nordrhein-Westfalen,Rheinland-Pfalz,Schleswig-Holstein,Saarland,Sachsen,Sachsen-Anhalt,Thueringen/Sachsen-Anhalt,Thueringen,Deutschland
date,,,,,,,,,,,,,,,,,
1881-01-01,-1.610000,-1.623333,0.286667,-1.580000,-0.500000,-2.226667,-0.800000,-0.810000,0.413333,0.483333,-1.776667,1.156667,-1.783333,-1.370000,-1.480000,-1.623333,-0.950000
1881-04-01,11.380000,11.363333,11.123333,10.216667,11.110000,10.430000,10.856667,10.856667,11.453333,11.486667,10.326667,11.763333,10.140000,11.180000,10.720000,10.140000,10.813333
1881-07-01,16.120000,16.100000,15.830000,15.243333,15.516667,15.226667,15.460000,15.460000,15.793333,15.766667,15.110000,15.980000,15.203333,15.853333,15.416667,14.866667,15.520000
1881-10-01,4.323333,4.303333,3.393333,2.573333,3.830000,4.390000,4.653333,4.656667,4.886667,4.130000,4.823333,4.220000,3.296667,4.186667,3.783333,3.273333,3.860000
1882-01-01,3.310000,3.296667,1.983333,0.980000,2.403333,3.056667,3.763333,3.763333,3.726667,2.586667,3.553333,3.013333,2.663333,3.323333,2.880000,2.323333,2.636667


In [10]:
# Downsample to yearly frequency
year_df = df.resample("YS").mean()
print(f"Yearly shape: {year_df.shape}")
year_df.head()

Yearly shape: (145, 17)


,Brandenburg/Berlin,Brandenburg,Baden-Wuerttemberg,Bayern,Hessen,Mecklenburg-Vorpommern,Niedersachsen,Niedersachsen/Hamburg/Bremen,Nordrhein-Westfalen,Rheinland-Pfalz,Schleswig-Holstein,Saarland,Sachsen,Sachsen-Anhalt,Thueringen/Sachsen-Anhalt,Thueringen,Deutschland
date,,,,,,,,,,,,,,,,,
1881-01-01,7.553333,7.535833,7.658333,6.613333,7.489167,6.955000,7.542500,7.540833,8.136667,7.966667,7.120833,8.280000,6.714167,7.462500,7.110000,6.664167,7.310833
1882-01-01,8.986667,8.972500,8.079167,7.328333,8.246667,8.540000,8.875000,8.878333,9.030000,8.551667,8.782500,8.785833,8.116667,8.812500,8.352500,7.772500,8.339167
1883-01-01,8.420833,8.408333,7.767500,6.847500,7.959167,7.948333,8.385833,8.388333,8.705833,8.261667,8.177500,8.510000,7.457500,8.315000,7.867500,7.305833,7.883333
1884-01-01,9.108333,9.095833,8.435833,7.523333,8.580000,8.727500,9.094167,9.097500,9.385833,8.936667,8.861667,9.177500,8.206667,8.936667,8.472500,7.885833,8.565000
1885-01-01,8.398333,8.388333,7.821667,7.043333,7.657500,7.680000,7.937500,7.936667,8.307500,8.006667,7.620833,8.296667,7.727500,8.074167,7.666667,7.155000,7.743333


In [11]:
# Upsample to daily frequency — new rows are NaN until filled
daily_df = df.resample("D").asfreq()
print(f"Daily shape: {daily_df.shape}")
print(f"Missing values: {daily_df.isnull().sum().sum()}")
daily_df.head(10)

Daily shape: (52808, 17)
Missing values: 868224


,Brandenburg/Berlin,Brandenburg,Baden-Wuerttemberg,Bayern,Hessen,Mecklenburg-Vorpommern,Niedersachsen,Niedersachsen/Hamburg/Bremen,Nordrhein-Westfalen,Rheinland-Pfalz,Schleswig-Holstein,Saarland,Sachsen,Sachsen-Anhalt,Thueringen/Sachsen-Anhalt,Thueringen,Deutschland
date,,,,,,,,,,,,,,,,,
1881-01-01,-5.54,-5.56,-4.89,-6.51,-5.68,-5.07,-4.55,-4.55,-4.21,-4.49,-4.06,-4.15,-6.22,-5.89,-6.28,-6.76,-5.36
1881-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1881-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1881-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1881-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1881-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1881-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1881-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1881-01-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Exercise.** Resample the CDC dataset to 10-year intervals and compute the maximum temperature recorded in each decade for `Deutschland`. Which decade was the warmest on record?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>5. Indexing and Slicing</h3>
</div>

With a `DatetimeIndex`, selecting a time range is just a slice. Pandas understands partial date strings, so you can write `'2015'` instead of `'2015-01-01'` and it will match the whole year. A few examples:

```python
df['2015']                  # all of 2015
df['2015-06']               # June 2015 only
df['2010':'2020']           # 2010 through 2020 inclusive
df[:'1900']                 # everything up to the end of 1900
df['2000':]                 # everything from 2000 onwards
```

In [12]:
# Last 10 years in the dataset
last_10y = df['2015':'2025']
print(f"Shape: {last_10y.shape}")
last_10y.head()

Shape: (128, 17)


,Brandenburg/Berlin,Brandenburg,Baden-Wuerttemberg,Bayern,Hessen,Mecklenburg-Vorpommern,Niedersachsen,Niedersachsen/Hamburg/Bremen,Nordrhein-Westfalen,Rheinland-Pfalz,Schleswig-Holstein,Saarland,Sachsen,Sachsen-Anhalt,Thueringen/Sachsen-Anhalt,Thueringen,Deutschland
date,,,,,,,,,,,,,,,,,
2015-01-01,2.80,2.79,1.69,1.08,1.85,2.75,3.03,3.04,2.86,1.89,2.98,1.85,2.09,2.82,2.31,1.65,2.19
2015-02-01,1.42,1.41,-0.66,-1.25,0.81,1.51,2.15,2.16,2.05,1.02,2.09,1.10,0.70,1.39,0.86,0.16,0.72
2015-03-01,5.60,5.59,5.17,4.59,4.97,5.25,5.61,5.61,5.54,5.37,5.36,5.74,4.95,5.51,5.02,4.39,5.18
2015-04-01,8.67,8.65,8.83,8.11,8.65,8.04,8.25,8.25,8.77,9.39,7.74,9.80,7.93,8.62,8.29,7.86,8.42
2015-05-01,12.72,12.70,13.09,12.66,12.49,11.31,11.66,11.65,12.28,12.91,10.72,13.12,12.62,12.62,12.42,12.16,12.34


In [14]:
# Selecting a single region
deutschland = df['Deutschland']
deutschland.head()

date
1881-01-01    -5.36
1881-02-01    -0.10
1881-03-01     2.61
1881-04-01     5.55
1881-05-01    11.81
Name: Deutschland, dtype: float64

In [16]:
# Selecting multiple regions
df[['Deutschland', 'Bayern', 'Schleswig-Holstein']].head()

,Deutschland,Bayern,Schleswig-Holstein
date,,,
1881-01-01,-5.36,-6.51,-4.06
1881-02-01,-0.10,-0.73,-1.81
1881-03-01,2.61,2.50,0.54
1881-04-01,5.55,5.01,5.02
1881-05-01,11.81,10.99,11.16


**Exercise.** Select all observations from January across all years (i.e. every row where the month is 1). What is the coldest January on record for `Deutschland`?

In [ ]:
# Hint: df.index.month gives you the month number for each row
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3>6. Rolling Windows</h3>
</div>

A rolling window computes a statistic over a sliding window of fixed size, moving one step at a time. It is one of the most common ways to smooth a noisy series or to generate lag-based features for a model.

The key parameters:

- `window`: the number of observations in the window.
- `min_periods`: the minimum number of non-NaN observations required to compute a result. If fewer observations are available (e.g. at the start of the series), the output is NaN. Setting it to `1` means the computation starts from the very first row; leaving it at the default (equal to `window`) means the first `window - 1` rows will be NaN.

In [ ]:
# 3-month rolling mean:  smooths out month-to-month noise
# min_periods=1 means we get a value even for the first two rows
rolling_3m = df.rolling(window=3, min_periods=1).mean()
rolling_3m.head(6)

,Brandenburg/Berlin,Brandenburg,Baden-Wuerttemberg,Bayern,Hessen,Mecklenburg-Vorpommern,Niedersachsen,Niedersachsen/Hamburg/Bremen,Nordrhein-Westfalen,Rheinland-Pfalz,Schleswig-Holstein,Saarland,Sachsen,Sachsen-Anhalt,Thueringen/Sachsen-Anhalt,Thueringen,Deutschland
date,,,,,,,,,,,,,,,,,
1881-01-01,-5.540000,-5.560000,-4.890000,-6.510000,-5.680000,-5.070000,-4.550000,-4.550000,-4.210000,-4.490000,-4.060000,-4.150000,-6.220000,-5.890,-6.280000,-6.760000,-5.360000
1881-02-01,-3.270000,-3.285000,-1.780000,-3.620000,-2.465000,-3.540000,-2.300000,-2.310000,-1.285000,-1.520000,-2.935000,-0.950000,-3.570000,-3.155,-3.310000,-3.505000,-2.730000
1881-03-01,-1.610000,-1.623333,0.286667,-1.580000,-0.500000,-2.226667,-0.800000,-0.810000,0.413333,0.483333,-1.776667,1.156667,-1.783333,-1.370,-1.480000,-1.623333,-0.950000
1881-04-01,2.090000,2.080000,4.126667,2.260000,3.353333,1.006667,2.596667,2.586667,3.963333,4.196667,1.250000,4.900000,1.760000,2.430,2.293333,2.106667,2.686667
1881-05-01,6.706667,6.693333,7.496667,6.166667,7.153333,5.536667,6.623333,6.620000,7.600000,7.796667,5.573333,8.276667,5.896667,6.760,6.423333,5.990000,6.656667
1881-06-01,11.380000,11.363333,11.123333,10.216667,11.110000,10.430000,10.856667,10.856667,11.453333,11.486667,10.326667,11.763333,10.140000,11.180,10.720000,10.140000,10.813333


In [ ]:
# 12-month rolling standard deviation: captures how variable each year is
# min_periods=12 means the first 11 rows will be NaN
rolling_12m_std = df.rolling(window=12, min_periods=12).std()
rolling_12m_std.head(15)

,Brandenburg/Berlin,Brandenburg,Baden-Wuerttemberg,Bayern,Hessen,Mecklenburg-Vorpommern,Niedersachsen,Niedersachsen/Hamburg/Bremen,Nordrhein-Westfalen,Rheinland-Pfalz,Schleswig-Holstein,Saarland,Sachsen,Sachsen-Anhalt,Thueringen/Sachsen-Anhalt,Thueringen,Deutschland
date,,,,,,,,,,,,,,,,,
1881-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1881-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1881-03-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1881-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1881-05-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1881-06-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1881-07-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1881-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1881-09-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Exercise.** Compute a 12-month centred rolling mean for `Deutschland`. A centred window (`center=True`) places the window symmetrically around each observation rather than looking only backwards. Does the result look different from the standard rolling mean? What does centering trade off?

In [ ]:
# Your solution here


---

You now have the core tools for loading and shaping time series data in pandas. The next notebook puts these to work visually: plotting the series, spotting seasonal patterns, and identifying anything unusual before you start building models.